# Spectral Gradient Methods — BB1 vs BB2

Interactive comparison of the **Barzilai–Borwein BB1 and BB2**
spectral gradient methods.

Select one of five objective functions, adjust the numerical parameters,
and compare convergence and spectral step-size behavior.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

root = Path.cwd()
if not (root / "gradient_methods").is_dir():
    for parent in root.parents:
        if (parent / "gradient_methods").is_dir():
            root = parent
            break

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from gradient_methods.spectral_gradient_solver import (
    solve_spectral_gradient,
    cost,
)


In [ ]:
def compare_spectral_methods(
    objective=5,
    dimension=10,
    initial_step=0.25,
    tolerance=1e-8,
    max_iterations=300,
):
    """Compare BB1 and BB2 using identical initial conditions."""

    x0 = np.zeros(dimension)

    results = {
        variant: solve_spectral_gradient(
            x0,
            variant=variant,
            objective=objective,
            initial_step=initial_step,
            max_iter=max_iterations,
            tol=tolerance,
        )
        for variant in ("BB1", "BB2")
    }

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))

    for variant, result in results.items():
        iterations = np.arange(len(result.cost_history))

        axes[0].plot(
            iterations,
            result.cost_history,
            label=variant,
            linewidth=2,
        )

        axes[1].semilogy(
            iterations,
            np.maximum(result.gradient_norms, 1e-16),
            label=variant,
            linewidth=2,
        )

        axes[2].semilogy(
            np.arange(1, len(result.step_history) + 1),
            result.step_history,
            label=variant,
            linewidth=2,
        )

    axes[0].set_title("Objective value")
    axes[0].set_xlabel("Iteration")
    axes[0].set_ylabel("J(x)")

    axes[1].set_title("Gradient norm")
    axes[1].set_xlabel("Iteration")
    axes[1].set_ylabel("||∇J(x)||")

    axes[2].set_title("Spectral step size")
    axes[2].set_xlabel("Iteration")
    axes[2].set_ylabel("Step size")

    for ax in axes:
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=9)

    fig.tight_layout()
    plt.show()

    rows = []

    for variant, result in results.items():
        rows.append(
            "<tr>"
            f"<td><b>{variant}</b></td>"
            f"<td>{result.iterations}</td>"
            f"<td>{'Yes' if result.converged else 'No'}</td>"
            f"<td>{result.cost_history[-1]:.6e}</td>"
            f"<td>{result.gradient_norms[-1]:.3e}</td>"
            "</tr>"
        )

    display(HTML(
        '<table style="border-collapse:collapse;width:100%;'
        'font-size:13px;text-align:center">'
        '<thead><tr style="background:#eef2f7">'
        '<th>Method</th><th>Iterations</th>'
        '<th>Converged</th><th>Final cost</th>'
        '<th>Gradient norm</th></tr></thead>'
        '<tbody>' + ''.join(rows) + '</tbody></table>'
    ))

    return results

In [ ]:
objective_widget = widgets.Dropdown(
    options=[
        ("J1 — Quadratic (ones)", 1),
        ("J2 — Quadratic (index target)", 2),
        ("J5 — Tridiagonal quadratic", 5),
        ("J6 — Quartic regularization", 6),
        ("JR — Rosenbrock", "R"),
    ],
    value=5,
    description="Objective:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="310px"),
)

dimension_widget = widgets.BoundedIntText(
    value=10,
    min=2,
    max=200,
    description="Dimension:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="180px"),
)

step_widget = widgets.BoundedFloatText(
    value=0.25,
    min=1e-8,
    max=100.0,
    step=0.05,
    description="Initial step:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="200px"),
)

tolerance_widget = widgets.Dropdown(
    options=[
        ("1e-4", 1e-4),
        ("1e-6", 1e-6),
        ("1e-8", 1e-8),
        ("1e-10", 1e-10),
    ],
    value=1e-8,
    description="Tolerance:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="175px"),
)

iterations_widget = widgets.BoundedIntText(
    value=300,
    min=0,
    max=5000,
    description="Max iterations:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="210px"),
)

run_button = widgets.Button(
    description="Run Comparison",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="170px", height="34px"),
)

output_widget = widgets.Output()

controls = widgets.VBox([
    widgets.HBox([
        objective_widget,
        dimension_widget,
        step_widget,
    ]),
    widgets.HBox([
        tolerance_widget,
        iterations_widget,
        run_button,
    ]),
])


status_widget = widgets.HTML(
    value=(
        '<div style="color:#526174;font-size:13px;margin:8px 0">'
        'Configure the parameters, then click '
        '<b>Run Comparison</b> to display the results.'
        '</div>'
    )
)


def mark_parameters_changed(change):
    status_widget.value = (
        '<div style="color:#a05a00;font-size:13px;'
        'margin:8px 0;font-weight:600">'
        '⚠️ Parameters changed. Click Run Comparison '
        'to update the results.'
        '</div>'
    )


for control in (
    objective_widget,
    dimension_widget,
    step_widget,
    tolerance_widget,
    iterations_widget,
):
    control.observe(mark_parameters_changed, names="value")


def run_comparison_updated(_=None):
    run_button.disabled = True

    try:
        with output_widget:
            clear_output(wait=True)

            try:
                compare_spectral_methods(
                    objective=objective_widget.value,
                    dimension=dimension_widget.value,
                    initial_step=step_widget.value,
                    tolerance=tolerance_widget.value,
                    max_iterations=iterations_widget.value,
                )
            except Exception as exc:
                print(f"Comparison failed: {exc}")
                status_widget.value = (
                    '<div style="color:#b42318;font-weight:600">'
                    'Comparison failed. Check the message below.'
                    '</div>'
                )
            else:
                status_widget.value = ""
    finally:
        run_button.disabled = False


run_button.on_click(run_comparison_updated)

display(controls, status_widget, output_widget)